# Week 06 - Day 05: Snowflake + dbt Hands-On Exercises
**Database:** `DBT_DB` | **Schemas:** `DEV_SCHEMA`, `SNAPSHOTS` | **Warehouse:** `DBT_WH`


### 1. Snowflake Setup (Warehouse, Database, and Role)


In [1]:
USE ROLE ACCOUNTADMIN;

CREATE WAREHOUSE IF NOT EXISTS dbt_wh WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE DATABASE IF NOT EXISTS dbt_db;
CREATE ROLE IF NOT EXISTS dbt_role;

GRANT USAGE ON WAREHOUSE dbt_wh TO ROLE dbt_role;
GRANT ALL ON DATABASE dbt_db TO ROLE dbt_role;
GRANT ALL ON ALL SCHEMAS IN DATABASE dbt_db TO ROLE dbt_role;
GRANT ROLE dbt_role TO USER ROSI;

USE ROLE dbt_role;
USE WAREHOUSE dbt_wh;
USE DATABASE dbt_db;
USE SCHEMA dev_schema;


ROLE,WAREHOUSE,DATABASE,SCHEMA
DBT_ROLE,DBT_WH,DBT_DB,DEV_SCHEMA


### 2. Exercise 1 — Staging Views (`STG_CUSTOMERS` & `STG_ORDERS`)


In [2]:
SELECT * FROM DBT_DB.DEV_SCHEMA.STG_CUSTOMERS LIMIT 10;


CUSTOMER_ID,CUSTOMER_NAME,FIRST_NAME,LAST_NAME,EMAIL,CITY,SIGNUP_DATE
1,William Jennings,William,Jennings,william.jennings.updated@example.com,Dublin,2025-01-29
2,Diane Newman,Diane,Newman,diane.newman2@example.com,Manchester,2025-11-06
3,Samuel Wright,Samuel,Wright,samuel.wright3@example.com,London,2024-08-12
4,Gillian Barnes,Gillian,Barnes,gillian.barnes4@example.com,Lisbon,2025-09-15
5,Nigel Edwards,Nigel,Edwards,nigel.edwards5@example.com,Glasgow,2025-11-23
6,Ms Rosemary Wright,Ms,Rosemary,ms.rosemary.wright6@example.com,Edinburgh,2026-02-07
7,Tracy Harrison,Tracy,Harrison,tracy.harrison7@example.com,Edinburgh,2025-03-16
8,Gillian Andrews,Gillian,Andrews,gillian.andrews8@example.com,Birmingham,2025-03-13
9,Janice Glover,Janice,Glover,janice.glover9@example.com,Lisbon,2026-03-06
10,Wayne Gough,Wayne,Gough,wayne.gough10@example.com,Manchester,2025-12-13


In [3]:
SELECT * FROM DBT_DB.DEV_SCHEMA.STG_ORDERS LIMIT 10;


ORDER_ID,CUSTOMER_ID,PRODUCT_ID,ORDER_DATE,QUANTITY,UNIT_PRICE,ORDER_STATUS,ORDER_AMOUNT
1001,21,105,2028-03-29,1,1999.00,paid,1999.00
1002,71,109,2025-03-04,2,2499.00,paid,4998.00
1003,86,105,2025-09-12,5,1999.00,paid,9995.00
1004,42,101,2026-10-14,2,249.00,pending,498.00
1005,24,107,2026-04-05,3,799.00,paid,2397.00
1006,40,106,2027-12-16,2,349.00,paid,698.00
1007,54,108,2026-09-10,2,149.00,paid,298.00
1008,90,109,2026-08-28,5,2499.00,paid,12495.00
1009,66,107,2027-06-27,5,799.00,paid,3995.00
1010,97,103,2025-05-22,5,1299.00,paid,6495.00


### 3. Exercise 1 — Final Mart Table (`FCT_ORDERS`)


In [4]:
SELECT * FROM DBT_DB.DEV_SCHEMA.FCT_ORDERS LIMIT 15;


ORDER_ID,CUSTOMER_ID,FIRST_NAME,LAST_NAME,ORDER_DATE,NET_AMOUNT
1001,21,Miss,Charlotte,2028-03-29,1999.00
1002,71,Fiona,Saunders-Mitchell,2025-03-04,4998.00
1003,86,Liam,Sykes,2025-09-12,9995.00
1004,42,Kayleigh,Mitchell,2026-10-14,498.00
1005,24,Leonard,Powell-Morris,2026-04-05,2397.00
1006,40,Mr,Philip,2027-12-16,698.00
1007,54,Catherine,Bates,2026-09-10,298.00
1008,90,Mr,Russell,2026-08-28,12495.00
1009,66,Janet,Wright,2027-06-27,3995.00
1010,97,Roger,Singh,2025-05-22,6495.00


### 4. Exercise 3 — Incremental Model Verification (`order_id` 1501 & 1502)


In [5]:
-- Verify total row count (500 initial + 2 incremental = 502) and inspect the 2 new rows:
SELECT * FROM DBT_DB.DEV_SCHEMA.FCT_ORDERS
WHERE order_id IN (1501, 1502)
ORDER BY order_id;


ORDER_ID,CUSTOMER_ID,FIRST_NAME,LAST_NAME,ORDER_DATE,NET_AMOUNT
1501,1,William,Jennings,2028-05-01,498.00
1502,2,Diane,Newman,2028-05-01,599.00


### 5. Exercise 4 — SCD Type 2 Snapshot History (`CUSTOMERS_SNAPSHOT`)


In [6]:
SELECT
    customer_id,
    customer_name,
    email,
    dbt_valid_from,
    dbt_valid_to
FROM DBT_DB.SNAPSHOTS.CUSTOMERS_SNAPSHOT
WHERE customer_id = 1
ORDER BY dbt_valid_from;


CUSTOMER_ID,CUSTOMER_NAME,EMAIL,DBT_VALID_FROM,DBT_VALID_TO
1,William Jennings,william.jennings1@example.com,2026-11-15 18:03:07.196000,2026-11-15 18:04:12.205000
1,William Jennings,william.jennings.updated@example.com,2026-11-15 18:04:12.205000,None
